# Navigation 공학 설계 — 실 sim 데이터 기반

본 자료는 **현재 실행 중인 ROS sim** 에서 직접 캡처한 데이터로 만들어졌다 (모든 그림은 `captures/` 의 실제 파일에서 생성).

**핵심 한 줄.** 단일 알고리즘·단일 평면·단일 호출로 풀 수 없는 결정을, 서로 다른 *시간 척도 · 정보 종류 · 실패 모델* 로 분해해 합성한다.

| # | 분리 차원 | 단독 한계 | 합성 효과 |
|---|---|---|---|
| 1 | 공간/시간 척도 | 빠른 반응 ↔ 전역 시야 양립 불가 | 계층 분해 (전역 ↔ 국소) |
| 2 | 정보 종류 | 한 layer 갱신이 다른 layer 침해 | 독립 layer + 소비 시점 합성 |
| 3 | 실패 모델 | 한 실패 = 전체 실패 | Behavior Tree 로 escalation |
| 4 | 미지 영역 | next-best-view 정의 불가 | 모델의 경계 = 정보 이득 |
| 5 | 실행 시간 | RPC 로 cancel·feedback 표현 불가 | 비동기 lifecycle 인터페이스 |

In [ ]:
import os, glob, subprocess, xml.etree.ElementTree as ET
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import font_manager as fm

# 한글 폰트 등록 (TTC 는 자동 인식 안 되므로 명시)
for cand in [
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    '/Library/Fonts/AppleGothic.ttf',
    'C:/Windows/Fonts/malgun.ttf',
]:
    if os.path.exists(cand):
        try:
            fm.fontManager.addfont(cand)
        except Exception:
            pass
for name in ['Noto Sans CJK KR', 'Noto Sans CJK JP', 'NanumGothic', 'AppleGothic', 'Malgun Gothic']:
    if any(f.name == name for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = name
        print('Using font:', name)
        break
plt.rcParams['axes.unicode_minus'] = False

CAP = 'captures'
FIG = 'figs'
os.makedirs(FIG, exist_ok=True)

---
## 1. 시간/공간 척도 분리 — Global ⊕ Local Costmap

- **문제.** *어디로 갈 것인가* (장기·전역) ↔ *지금 무엇을 할 것인가* (실시간·국소) 는 본질적으로 다른 척도
- **아이디어.** 두 결정을 **다른 해상도·범위·갱신 주기** 의 비용 평면으로 분리, 전역 → 참조 / 국소 → 동역학 회피

| 평면 | 좌표계 | 범위 | 입력 | 알고리즘 |
|---|---|---|---|---|
| Global | `map` | 환경 전체 (수십 m) | SLAM 지도 + 누적 관측 | 그래프 탐색 (NavFn/Smac) |
| Local | `odom` | 로봇 근방 3 m × 3 m | 실시간 LiDAR/Depth | DWB rollout |

**왜 합성?** 단일 평면이면 *전역 cover ↔ 실시간 갱신* 둘 중 하나만 만족. 분리하면 각자 책임만 보장 → 전역 경로가 국소 트래커의 **참조 신호** 가 됨.

In [ ]:
# 실제 캡처된 두 costmap 을 같은 미터 단위로 스케일 비교
g = np.load(f'{CAP}/costmap_global.npz')
l = np.load(f'{CAP}/costmap_local.npz')

def costmap_image(arr):
    # OccupancyGrid 값: -1 unknown, 0 free, 1~99 inflated, 100 lethal
    img = np.zeros((*arr.shape, 3), dtype=float)
    img[arr == -1] = [0.6, 0.6, 0.6]      # unknown gray
    img[arr == 0]  = [1.0, 1.0, 1.0]      # free white
    inflated = (arr > 0) & (arr < 100)
    img[inflated] = np.stack([np.ones(inflated.sum()),
                              1.0 - arr[inflated]/100,
                              np.zeros(inflated.sum())], axis=-1)  # yellow→red
    img[arr >= 100] = [0.0, 0.0, 0.0]     # lethal black
    return img

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, src, label in zip(axes, [g, l], ['Global Costmap (frame=map)', 'Local Costmap (frame=odom)']):
    img = costmap_image(src['grid'])
    res = float(src['resolution'])
    w_m = src['grid'].shape[1] * res
    h_m = src['grid'].shape[0] * res
    ax.imshow(img, origin='lower', extent=[0, w_m, 0, h_m])
    ax.set_title(f'{label}\n{src["grid"].shape[1]}×{src["grid"].shape[0]} @ {res:.2f} m  →  {w_m:.1f} m × {h_m:.1f} m',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_aspect('equal')

fig.suptitle('동일 데모: Global = 환경 전체 / Local = 로봇 근방 3m 윈도우', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG}/01_costmap_scales.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Global 면적: {g["grid"].shape[1]*g["resolution"]:.1f} × {g["grid"].shape[0]*g["resolution"]:.1f} m  ({g["grid"].size} cells)')
print(f'Local 면적:  {l["grid"].shape[1]*l["resolution"]:.1f} × {l["grid"].shape[0]*l["resolution"]:.1f} m  ({l["grid"].size} cells)')
print(f'두 평면이 동일 해상도(0.05 m)지만, 범위·좌표계·갱신 주기가 다름 — 한 평면으로 둘 다 풀 수 없음')

---
## 2. 비용 평면의 Layered 합성 — 실제 config

- **문제.** 정적 지도 / 실시간 관측 / 안전 마진은 출처·갱신 주기·신뢰도가 다름. 한 평면에 합치면 한 종류의 갱신이 다른 종류를 침해.
- **아이디어.** 독립 layer 로 유지, *소비 시점에만* 합성된 단일 비용 평면을 플래너에 노출.

**우리 sim 의 실제 layer 조합** (`nav2_params_d456.yaml`):
```
global_costmap:  plugins: [static_layer, obstacle_layer, inflation_layer]
local_costmap:   plugins: [voxel_layer, inflation_layer]
                 inflation_radius: 0.55,  cost_scaling_factor: 3.0,  robot_radius: 0.22
```

아래 그림은 **실제 캡처된 global costmap** 의 cell 분포를 의미별로 시각화한 것 — free / unknown / inflated / lethal 셀이 모두 같은 단일 평면에 합성되어 있다.

In [ ]:
g = np.load(f'{CAP}/costmap_global.npz')
arr = g['grid']
res = float(g['resolution'])

fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1])
ax = fig.add_subplot(gs[0, 0])
ax.imshow(costmap_image(arr), origin='lower',
          extent=[0, arr.shape[1]*res, 0, arr.shape[0]*res])
ax.set_title(f'Composite Global Costmap (실제 캡처)\nstatic ⊕ obstacle ⊕ inflation', fontsize=12, fontweight='bold')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_aspect('equal')

n_unknown = int((arr == -1).sum())
n_free = int((arr == 0).sum())
n_inflated = int(((arr > 0) & (arr < 100)).sum())
n_lethal = int((arr >= 100).sum())
total = arr.size

ax2 = fig.add_subplot(gs[0, 1])
labels = ['Unknown\n(-1)', 'Free\n(0)', 'Inflated\n(1~99)', 'Lethal\n(100)']
vals = [n_unknown, n_free, n_inflated, n_lethal]
colors = ['#999999', '#ffffff', '#ff7f00', '#000000']
bars = ax2.bar(labels, vals, color=colors, edgecolor='black')
ax2.set_ylabel('cells')
ax2.set_title('Cell 분포 — 한 평면에 4가지 의미 공존', fontsize=11, fontweight='bold')
for b, v in zip(bars, vals):
    ax2.text(b.get_x() + b.get_width()/2, v, f'{v}\n({v/total*100:.1f}%)',
             ha='center', va='bottom', fontsize=9)
ax2.set_ylim(0, max(vals)*1.18)

plt.tight_layout()
plt.savefig(f'{FIG}/02_costmap_layers.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'전체 {total} cells = unknown {n_unknown} + free {n_free} + inflated {n_inflated} + lethal {n_lethal}')
print('→ 각 의미는 다른 layer 가 책임지지만, 플래너가 보는 합성 평면은 단일.')

---
## 3. 정책의 명시적 트리 표현 — 실제 BT XML

- **문제.** 부분 실패 (계획 실패, 갇힘) 가 빈번. if/else 코드로 묻으면 정책 변경 = 코드 변경.
- **아이디어.** **Algorithm = mechanism, Tree = policy** — 알고리즘은 단위 행동만 제공, 트리가 그 조합·실패 처리를 선언적으로 표현.

아래 그림은 우리 sim 이 사용하는 실제 `navigate_to_pose_w_replanning_and_recovery.xml` 을 그대로 파싱·렌더링한 것.

In [ ]:
tree = ET.parse(f'{CAP}/bt_navigate_to_pose.xml')
root_xml = tree.getroot()
bt = root_xml.find('BehaviorTree')

control_tags = {'RecoveryNode', 'PipelineSequence', 'RateController',
                'ReactiveFallback', 'RoundRobin', 'Sequence', 'Fallback',
                'BehaviorTree', 'root'}
action_color = {
    'ComputePathToPose': '#a8e6a3',
    'FollowPath': '#a8e6a3',
    'ClearEntireCostmap': '#ffd6a5',
    'Spin': '#ffadad',
    'Wait': '#ffadad',
    'BackUp': '#ffadad',
    'GoalUpdated': '#fdffb6',
}

lines = ['digraph BT {', '  rankdir=TB;', '  graph [splines=ortho, nodesep=0.25];',
         '  node [shape=box, style="rounded,filled", fontname="Sans", fontsize=11];']
ctr = [0]
def walk(n, parent=None):
    nid = f'n{ctr[0]}'; ctr[0] += 1
    label = n.tag
    if 'name' in n.attrib:
        label += '\\n' + n.attrib['name']
    if n.tag in control_tags:
        color = '#cce0ff'
    else:
        color = action_color.get(n.tag, '#ffffff')
    lines.append(f'  {nid} [label="{label}", fillcolor="{color}"];')
    if parent is not None:
        lines.append(f'  {parent} -> {nid};')
    for child in n:
        walk(child, nid)

walk(bt)
lines.append('}')
with open(f'{FIG}/bt.dot', 'w') as f:
    f.write('\n'.join(lines))
subprocess.run(['dot', '-Tpng', f'{FIG}/bt.dot', '-o', f'{FIG}/03_bt_real.png'], check=True)

from IPython.display import Image, display
display(Image(f'{FIG}/03_bt_real.png'))
print('파란색 = control flow (Sequence/Fallback/Recovery)')
print('초록색 = 주행 액션 / 주황색 = 클리어 / 빨강 = 복구 행동')
print('실패 시 RecoveryFallback → RoundRobin 으로 [Clear → Spin → Wait → BackUp] 순환')

---
## 4. Frontier 기반 자율 탐사 — 실제 SLAM /map 에서 추출

- **문제.** 사전 지도 없이 탐사할 때 *어디로 다음에 가야 하는가* (next-best-view) 는 완전 모델 없으면 정의 불가.
- **아이디어.** 탐사 = **free 셀과 unknown 셀의 경계 (frontier) 검출** — 모델이 *아는 것의 형태* 자체가 정보 이득의 정의.

아래 그림은 우리 sim 의 실제 `/map` (SLAM 출력) 에서 free↔unknown 경계를 직접 계산한 결과. 별도 알고리즘 없이 "안다 ↔ 모른다" 의 인접 관계만으로 후보 목적지가 도출됨.

In [ ]:
from scipy.ndimage import binary_dilation

m = np.load(f'{CAP}/map_slam.npz')
grid = m['grid']
res = float(m['resolution'])

free = (grid == 0)
unknown = (grid == -1)
occ = (grid > 50)
frontier = free & binary_dilation(unknown, iterations=1)

img = np.full((*grid.shape, 3), 0.6)  # unknown gray
img[free] = [1.0, 1.0, 1.0]
img[occ] = [0.0, 0.0, 0.0]
img[frontier] = [0.1, 0.7, 1.0]  # cyan frontier

fig, ax = plt.subplots(figsize=(7, 11))
ax.imshow(img, origin='lower', extent=[0, grid.shape[1]*res, 0, grid.shape[0]*res])
ax.set_title(f'실제 SLAM /map ({grid.shape[1]}×{grid.shape[0]} @ {res:.2f}m)\n'
             f'경계(frontier) = free 셀 중 unknown 과 인접한 것',
             fontsize=11, fontweight='bold')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_aspect('equal')

n_free = int(free.sum()); n_unk = int(unknown.sum()); n_occ = int(occ.sum()); n_fr = int(frontier.sum())
ax.text(0.02, 0.98, f'free   {n_free:,}\nunknown {n_unk:,}\noccupied {n_occ:,}\nfrontier {n_fr:,} (cyan)',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.savefig(f'{FIG}/04_frontier_real.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Frontier {n_fr} cells = next-best-view 후보 (정보 이득 + 거리 비용으로 정렬)')
print('→ 탐사 알고리즘은 "goal 후보" 만 정의, Nav 스택이 그 goal 까지 안전 주행 — 책임 분리')

---
## 5. 비동기 Goal-Oriented 액션 — 실제 인터페이스

- **문제.** 주행은 수십 초 단위 long-running task. RPC 로는 cancel · feedback · 진행률 표현 불가.
- **아이디어.** Lifecycle 자체를 인터페이스의 일부로. 목표 송신 → feedback push → cancel 가능 → result 통지.

아래는 실제 sim 의 `/navigate_to_pose` 액션 메타데이터 (캡처본).

In [ ]:
with open(f'{CAP}/action_navigate_to_pose.txt') as f:
    text = f.read()
print(text)

In [ ]:
# Sync RPC vs Async action 타임라인 비교 — 위 메타데이터의 의미를 도식으로
fig, axes = plt.subplots(2, 1, figsize=(14, 5))

for ax in axes:
    ax.set_xlim(0, 20); ax.set_ylim(-0.3, 4)
    ax.set_yticks([1, 3]); ax.set_yticklabels(['Server', 'Client'], fontsize=10)
    ax.set_xlabel('Time')
    ax.plot([0, 20], [3, 3], 'k-', alpha=0.3); ax.plot([0, 20], [1, 1], 'k-', alpha=0.3)

ax = axes[0]
ax.set_title('Sync RPC — cancel/feedback 표현 불가', fontsize=11, fontweight='bold')
ax.annotate('', xy=(2, 1), xytext=(1, 3), arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.text(1, 2.4, 'call', fontsize=9, color='blue')
ax.add_patch(patches.Rectangle((1, 2.7), 14, 0.5, color='red', alpha=0.3))
ax.text(8, 2.95, 'BLOCKED', ha='center', fontsize=10, color='darkred', fontweight='bold')
ax.add_patch(patches.Rectangle((2, 0.8), 13, 0.5, color='green', alpha=0.3))
ax.text(8, 1.05, 'navigating...', ha='center', fontsize=9)
ax.annotate('', xy=(16, 3), xytext=(15, 1), arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
ax.text(15, 1.7, 'return', fontsize=9, color='green')

ax = axes[1]
ax.set_title('Async Action (실제 /navigate_to_pose) — feedback push + cancel', fontsize=11, fontweight='bold')
ax.annotate('', xy=(2, 1), xytext=(1, 3), arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.text(1, 2.4, 'send_goal\n(PoseStamped)', fontsize=8, color='blue')
for t in [4, 6, 8, 10]:
    ax.annotate('', xy=(t, 3), xytext=(t, 1), arrowprops=dict(arrowstyle='->', color='gray', lw=1))
    ax.text(t, 2.1, 'fb', fontsize=7, color='gray', ha='center')
ax.text(7, 3.45, 'feedback: current_pose, distance_remaining, eta', fontsize=8, color='gray', ha='center')
ax.add_patch(patches.Rectangle((2, 0.8), 11, 0.5, color='green', alpha=0.3))
ax.text(7, 1.05, 'navigating...', ha='center', fontsize=9)
ax.annotate('', xy=(13, 1), xytext=(12, 3), arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax.text(11.7, 2.4, 'cancel', fontsize=9, color='red')
ax.add_patch(patches.Rectangle((13, 0.8), 1.5, 0.5, color='orange', alpha=0.4))
ax.text(13.7, 1.05, 'safe stop', ha='center', fontsize=8)
ax.annotate('', xy=(15, 3), xytext=(14.5, 1), arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
ax.text(14.7, 1.7, 'result\n(cancelled)', fontsize=8, color='green')

plt.tight_layout()
plt.savefig(f'{FIG}/05_async_timeline.png', dpi=130, bbox_inches='tight')
plt.show()

print('인터페이스 자체에 lifecycle (goal → feedback* → result | cancelled) 이 박혀 있음')
print('→ 클라이언트는 polling 없이 push 받음, 서버는 cancel 시 graceful stop 보장')

---
## (Bonus) TF Tree — 실제 캡처

주행이 일관성 있게 동작하려면 모든 비용 평면 / 센서 / 명령이 **공통 좌표계** 위에 있어야 한다. `map → odom → base_link → 센서` 의 단방향 트리로 표현.

In [ ]:
from IPython.display import Image, display
if os.path.exists(f'{FIG}/tf_frames.png'):
    display(Image(f'{FIG}/tf_frames.png'))
else:
    print('TF tree PNG not found — run scripts/render_tf.sh')

---
## 정리 슬라이드

| # | 분리 차원 | 적용 패턴 |
|---|---|---|
| 1 | 공간/시간 척도 | Hierarchical planning (Global ↔ Local) |
| 2 | 정보 종류 | Layered cost representation |
| 3 | 실패 모델 | Behavior Tree (policy/mechanism 분리) |
| 4 | 미지 영역 | Frontier (모델의 경계 = 정보 이득) |
| 5 | 실행 시간 | Async action (forward goal + backward feedback/result) |

> **"단일 평면·단일 알고리즘·단일 호출로 풀 수 없는 결정을, 서로 다른 차원으로 분해해 합성한다."**

모든 그림은 본 sim 에서 직접 캡처한 데이터에서 생성:
- `captures/costmap_global.npz` — `/global_costmap/costmap` (128×413 @ 5cm)
- `captures/costmap_local.npz` — `/local_costmap/costmap` (60×60 @ 5cm)
- `captures/map_slam.npz` — SLAM `/map` (414×128 @ 5cm)
- `captures/bt_navigate_to_pose.xml` — Nav2 가 실제 사용하는 BT
- `captures/action_navigate_to_pose.txt` — 실제 액션 메타데이터
- `captures/tf_frames.pdf` — 실 TF 트리 (`view_frames`)